In [ ]:
!pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [ ]:
!pip install langchainhub

previously, it was *from langchain import hub* but now *hub* was removed from core *langchain* package. so, now it became. *import lanchainhub as hub*

In [ ]:
!pip install langchain_text_splitters

Previously, it was *from langchain.text_splitter import RecursiveCharacterTextSplitter* . Now, *text_splitter* is removed from core *langchain* package and to its own package *langchain_text_splitters*

In [ ]:
import os
os.environ['LANGCHAIN_TRACKING_V2'] ='true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = 'your-langsmith-key-here'

In [ ]:
os.environ['OPENAI_API_KEY']= 'your-open-ai-key-here'

In [ ]:
import bs4
import langchainhub as hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

WebBaseLoader → bs4 cleans HTML → RecursiveCharacterTextSplitter chunks it
→ OpenAIEmbeddings converts chunks → Chroma stores them
→ User asks question → Chroma retrieves relevant chunks
→ RunnablePassthrough passes question through → hub.pull provides the prompt
→ ChatOpenAI generates answer → StrOutputParser returns clean text.



When WebBaseLoader makes HTTP requests to websites, it sends a User-Agent header identifying who is making the request. Right now it's sending a generic/blank identifier, which some websites block or flag as suspicious bot traffic.
It won't break anything if you ignore it, but setting it is good practice — especially if you're scraping sites that enforce bot protection.

In [ ]:
import os
os. environ['USER_AGENT'] ='MyRAGapp/1.0'

In [ ]:
#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths= ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)
docs=loader.load()

#split
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits=text_splitter.split_documents(docs)

#Embed
vector_store=Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever=vector_store.as_retreiver()

#### RETRIEVAL AND GENERATION ####

# prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
llm=ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Post-processing
def format_docs(docs):
  return "\n\n".join()


# Chain
rag_chain = (
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)

# Question
rag_chain.invoke('what is Task Decomposition')

NameError: name 'Chroma' is not defined

***temperature=0*** means deterministic — no randomness, same question always gives same answer (good for factual Q&A)